# Sesi 2 — NusaMart Profiling Starter

Notebook ini dipakai bersama **Workspace Starter yang sama dari Sesi 1** dan **NusaMart Raw Practice Package**.

## Workflow Sesi 2

**1. Mixed-format Intake → 2. Understand Structure → 3. Grain & Candidate Key → 4. Data Quality Criteria → 5. Manual Profiling → 6. Antigravity Profiling → 7. Manual Verification → 8. Verified Finding Register**

Working Paper:
- `01_Source_Structure`
- `02_Grain_Key`
- `03_DQ_Criteria`
- `04_Manual_Profiling`
- `05_Verified_Findings`

Output resmi Sesi 2 tetap:
1. **Profiling Report Hasil Antigravity** → `outputs/sesi_02/profiling_report.md`
2. **Daftar Temuan Terverifikasi Manual** → `05_Verified_Findings`

## Guardrails
1. Jangan melakukan cleaning.
2. Jangan melakukan final relational join.
3. Jangan mengubah raw/source.
4. Repeated key ≠ otomatis exact duplicate.
5. Missing ≠ otomatis harus diimpute.
6. Unusual value ≠ otomatis error.
7. Antigravity menghasilkan **candidate findings**; peserta tetap wajib melakukan manual verification.

> Technical profiling helpers sudah tersedia. Peserta mengisi **grain/key, criteria confirmation, pilihan checks, dan verification judgment**. Mentor tidak perlu mengirim query/code tambahan agar practice selesai.

## 0. Setup & Project Root

Starter mencari project root berdasarkan folder `data/raw/nusamart/`.

Jika auto-detect gagal, isi `PROJECT_ROOT_OVERRIDE`.

In [1]:
print("hello world")

hello world


### Source/Path Cek

In [1]:
# Cek keberadaan 4 file raw source
from pathlib import Path
# ...

In [2]:
# Load ke DataFrame
import pandas as pd
# ...

In [5]:
import os
from pathlib import Path

# Kita langsung tembak lokasi absolut folder data Anda
data_dir = Path(r"C:\Bootcamp\Bootcamp_Data_Analyst\data\raw\nusamart")

print("Mengecek folder di:", data_dir)

files = [
    "nusamart_orders.xlsx",
    "nusamart_order_items.csv",
    "nusamart_customers.json",
    "nusamart_payments.txt"
]

for f in files:
    path = data_dir / f
    print(f"{'✅' if path.exists() else '❌'} {f} — {'found' if path.exists() else 'not found'}")


Mengecek folder di: C:\Bootcamp\Bootcamp_Data_Analyst\data\raw\nusamart
✅ nusamart_orders.xlsx — found
✅ nusamart_order_items.csv — found
✅ nusamart_customers.json — found
✅ nusamart_payments.txt — found


In [1]:
import os
from pathlib import Path

print("Current working directory:", os.getcwd())

# Naik DUA tingkat: keluar dari 'sesi_02' dan keluar dari 'notebooks'
os.chdir("../..")  

print("New working directory:", os.getcwd())

from pathlib import Path
# Sesuaikan nama folder sesuai gambar Anda
data_dir = Path("data\raw\nusamart")

files = [
    "nusamart_orders.xlsx",
    "nusamart_order_items.csv",
    "nusamart_customers.json",
    "nusamart_payments.txt"
]

for f in files:
    path = data_dir / f
    print(f"{'✅' if path.exists() else '❌'} {f} — {'found' if path.exists() else 'not found'}")


Current working directory: c:\Bootcamp\Bootcamp_Data_Analyst\notebooks\sesi_02
New working directory: c:\Bootcamp\Bootcamp_Data_Analyst
❌ nusamart_orders.xlsx — not found
❌ nusamart_order_items.csv — not found
❌ nusamart_customers.json — not found
❌ nusamart_payments.txt — not found


In [1]:
import os
from pathlib import Path

print("Current working directory:", os.getcwd())

# Naik DUA tingkat: keluar dari 'sesi_02' dan keluar dari 'notebooks'
os.chdir("../..")  

print("New working directory:", os.getcwd())

from pathlib import Path
# Sesuaikan nama folder sesuai gambar Anda
data_dir = Path("data/raw/nusamart")

files = [
    "nusamart_orders.xlsx",
    "nusamart_order_items.csv",
    "nusamart_customers.json",
    "nusamart_payments.txt"
]

for f in files:
    path = data_dir / f
    print(f"{'✅' if path.exists() else '❌'} {f} — {'found' if path.exists() else 'not found'}")


Current working directory: c:\Bootcamp\Bootcamp_Data_Analyst\notebooks\sesi_02
New working directory: c:\Bootcamp\Bootcamp_Data_Analyst
✅ nusamart_orders.xlsx — found
✅ nusamart_order_items.csv — found
✅ nusamart_customers.json — found
✅ nusamart_payments.txt — found


In [2]:
from __future__ import annotations

from hashlib import sha256
from pathlib import Path
import json
import re

import pandas as pd
try:
    import duckdb
    DUCKDB_AVAILABLE = True
except ImportError:
    duckdb = None
    DUCKDB_AVAILABLE = False
from IPython.display import display

PROJECT_ROOT_OVERRIDE = Path(r"C:\Bootcamp\Bootcamp_Data_Analyst") # contoh: Path(r"C:\...\Bootcamp_Data_Analyst")
RAW_REL = Path("data/raw/nusamart")

def resolve_project_root(override=None) -> Path:
    if override is not None:
        root = Path(override).expanduser().resolve()
        if not (root / RAW_REL).exists():
            raise FileNotFoundError(
                f"PROJECT_ROOT_OVERRIDE tidak valid: {root}\n"
                f"Folder yang dicari: {root / RAW_REL}"
            )
        return root

    start = Path.cwd().resolve()
    candidates = [start, *start.parents]

    for candidate in candidates:
        if (candidate / RAW_REL).exists():
            return candidate

    for candidate in candidates:
        nested = candidate / "Bootcamp_Data_Analyst"
        if (nested / RAW_REL).exists():
            return nested.resolve()

    raise FileNotFoundError(
        "Project root tidak ditemukan.\n"
        "Pastikan Workspace Starter memiliki data/raw/nusamart/.\n"
        "Jika notebook dibuka dari lokasi lain, isi PROJECT_ROOT_OVERRIDE."
    )

ROOT = resolve_project_root(PROJECT_ROOT_OVERRIDE)
RAW = ROOT / RAW_REL
OUTPUT_DIR = ROOT / "outputs" / "sesi_02"
PROFILING_REPORT = OUTPUT_DIR / "profiling_report.md"

FILES = {
    "orders": RAW / "nusamart_orders.xlsx",
    "order_items": RAW / "nusamart_order_items.csv",
    "customers": RAW / "nusamart_customers.json",
    "payments": RAW / "nusamart_payments.txt",
}

missing_files = [str(path) for path in FILES.values() if not path.exists()]
if missing_files:
    raise FileNotFoundError(
        "RAW PACKAGE CHECK FAILED. File berikut belum ditemukan:\n- "
        + "\n- ".join(missing_files)
    )

print("PROJECT_ROOT :", ROOT)
print("RAW_DIR      :", RAW)
print("PROFILING REPORT EXPECTED:", PROFILING_REPORT)

PROJECT_ROOT : C:\Bootcamp\Bootcamp_Data_Analyst
RAW_DIR      : C:\Bootcamp\Bootcamp_Data_Analyst\data\raw\nusamart
PROFILING REPORT EXPECTED: C:\Bootcamp\Bootcamp_Data_Analyst\outputs\sesi_02\profiling_report.md


## 1. Mixed-format Intake

Loader membaca **XLSX + CSV + JSON + TXT** menjadi empat logical datasets.

Fokus latihan adalah memahami dan memprofile data, bukan membuat parser dari nol.

In [ ]:
# !pip install openpyxl

Defaulting to user installation because normal site-packages is not writeable

   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\lucih\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [3]:
def read_csv_robust(path: Path) -> pd.DataFrame:
    last_error = None
    for enc in ("utf-8-sig", "utf-8", "latin1"):
        try:
            return pd.read_csv(path, dtype=object, encoding=enc)
        except UnicodeDecodeError as err:
            last_error = err
    raise last_error

def load_orders(path: Path) -> pd.DataFrame:
    xls = pd.ExcelFile(path, engine="openpyxl")
    if "orders_raw" in xls.sheet_names:
        sheet = "orders_raw"
    elif len(xls.sheet_names) == 1:
        sheet = xls.sheet_names[0]
    else:
        raise ValueError(
            f"Sheet orders tidak ditemukan. Available sheets: {xls.sheet_names}"
        )
    return pd.read_excel(path, sheet_name=sheet, dtype=object, engine="openpyxl")

def load_order_items(path: Path) -> pd.DataFrame:
    return read_csv_robust(path)

def load_customers(path: Path) -> pd.DataFrame:
    raw = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(raw, list):
        raise ValueError("customers.json harus berisi list records.")

    rows = []
    for obj in raw:
        if "profile" in obj or "contact" in obj:
            rows.append({
                "customer_id": obj.get("customer_id"),
                "customer_name": obj.get("profile", {}).get("customer_name"),
                "city": obj.get("profile", {}).get("city"),
                "customer_segment": obj.get("profile", {}).get("customer_segment"),
                "join_date": obj.get("profile", {}).get("join_date"),
                "email": obj.get("contact", {}).get("email"),
                "phone": obj.get("contact", {}).get("phone"),
                "customer_status": obj.get("customer_status"),
            })
        else:
            rows.append(obj)

    return pd.DataFrame(rows)

def load_payments(path: Path) -> pd.DataFrame:
    rows = []
    for line_no, line in enumerate(path.read_text(encoding="utf-8").splitlines(), start=1):
        if not line.strip():
            continue

        record = {}
        for part in line.split("|"):
            part = part.strip()
            if "=" not in part:
                raise ValueError(f"payments.txt line {line_no} malformed: {part!r}")
            key, value = part.split("=", 1)
            record[key.strip().upper()] = value.strip()

        rows.append({
            "payment_id": record.get("PAYMENT_ID"),
            "order_id": record.get("ORDER_ID"),
            "payment_date": record.get("DATE"),
            "payment_method": record.get("METHOD"),
            "payment_amount": record.get("AMOUNT"),
            "payment_status": record.get("STATUS"),
        })
    return pd.DataFrame(rows)

def load_all_raw() -> dict[str, pd.DataFrame]:
    return {
        "orders": load_orders(FILES["orders"]),
        "order_items": load_order_items(FILES["order_items"]),
        "customers": load_customers(FILES["customers"]),
        "payments": load_payments(FILES["payments"]),
    }

datasets = load_all_raw()

MIN_REQUIRED_COLUMNS = {
    "orders": {"order_id", "order_date"},
    "order_items": {"order_id", "quantity"},
    "customers": {"customer_id", "customer_segment"},
    "payments": {"payment_id", "order_id", "payment_date", "payment_amount"},
}

schema_issues = []
for name, required in MIN_REQUIRED_COLUMNS.items():
    missing = sorted(required - set(datasets[name].columns))
    if missing:
        schema_issues.append(f"{name}: missing columns {missing}")

intake_schema_pass = len(schema_issues) == 0

if not intake_schema_pass:
    raise ValueError("SCHEMA CHECK FAILED:\n- " + "\n- ".join(schema_issues))

for name, df in datasets.items():
    print(f"{name:12} -> shape={df.shape}")

print("RAW LOAD & MINIMUM SCHEMA CHECK: PASS")

# DuckDB tetap menjadi tool utama kelas bila tersedia.
# Fallback pandas membuat Starter tetap dapat dibuka/dry-run tanpa crash teknis.
con = None
if DUCKDB_AVAILABLE:
    con = duckdb.connect()
    for table_name, df in datasets.items():
        con.register(table_name, df)
    print("DUCKDB REGISTRATION: PASS — orders, order_items, customers, payments")
else:
    print("DUCKDB REGISTRATION: SKIPPED — package duckdb tidak tersedia di environment ini.")
    print("Profiling helpers tetap dapat berjalan dengan pandas fallback.")

orders       -> shape=(320, 8)
order_items  -> shape=(763, 8)
customers    -> shape=(215, 8)
payments     -> shape=(337, 6)
RAW LOAD & MINIMUM SCHEMA CHECK: PASS
DUCKDB REGISTRATION: PASS — orders, order_items, customers, payments


In [7]:
# 1. Cek file Excel (.xlsx)
print("=== TIPE DATA: ORDERS (Excel) ===")
df_orders = pd.read_excel(FILES["orders"])
print(df_orders.dtypes)
print("\n" + "="*40 + "\n")

# 2. Cek file CSV (.csv)
print("=== TIPE DATA: ORDER ITEMS (CSV) ===")
df_items = pd.read_csv(FILES["order_items"])
print(df_items.dtypes)
print("\n" + "="*40 + "\n")

# 3. Cek file JSON (.json)
print("=== TIPE DATA: CUSTOMERS (JSON) ===")
df_customers = pd.read_json(FILES["customers"])
print(df_customers.dtypes)
print("\n" + "="*40 + "\n")

# 4. Cek file TXT (.txt) 
# Catatan: File teks biasanya dipisahkan oleh karakter tertentu, misal Tab (\t) atau Koma (,)
print("=== TIPE DATA: PAYMENTS (TXT) ===")
try:
    # Kita coba baca dengan anggapan pemisah antar kolomnya adalah Tab
    df_payments = pd.read_csv(FILES["payments"], sep="\t") 
    print(df_payments.dtypes)
except Exception as e:
    print("Gagal membaca file TXT secara otomatis, error:", e)


=== TIPE DATA: ORDERS (Excel) ===
order_id            str
customer_id         str
order_date       object
sales_channel       str
order_status        str
shipping_city       str
shipping_fee      int64
order_total       int64
dtype: object


=== TIPE DATA: ORDER ITEMS (CSV) ===
order_item_id         str
order_id              str
product_id            str
product_category      str
quantity            int64
unit_price            str
discount_amount       str
line_amount         int64
dtype: object


=== TIPE DATA: CUSTOMERS (JSON) ===
customer_id           str
profile            object
contact            object
customer_status       str
dtype: object


=== TIPE DATA: PAYMENTS (TXT) ===
PAYMENT_ID=PAY0001 | ORDER_ID=ORD0001 | DATE=2026-03-27 | METHOD=QRIS | AMOUNT=564000 | STATUS=Paid    str
dtype: object


## 2. Understand Structure + Raw Integrity Snapshot

Isi `01_Source_Structure` berdasarkan hasil aktual.

Physical representation:
- Orders → XLSX → row
- Order Items → CSV → row
- Customers → JSON → object
- Payments → TXT → line

Hash raw files dicatat sekarang dan diperiksa lagi di akhir S2.

In [4]:
SOURCE_STRUCTURE = {
    "orders": {"format": "XLSX", "record_structure": "row"},
    "order_items": {"format": "CSV", "record_structure": "row"},
    "customers": {"format": "JSON", "record_structure": "object"},
    "payments": {"format": "TXT", "record_structure": "line"},
}

def file_sha256(path: Path) -> str:
    digest = sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

raw_hash_before = {name: file_sha256(path) for name, path in FILES.items()}

source_rows = []
for name, df in datasets.items():
    meta = SOURCE_STRUCTURE[name]
    source_rows.append({
        "dataset": name,
        "physical_file": FILES[name].name,
        "format": meta["format"],
        "record_structure": meta["record_structure"],
        "records": len(df),
        "columns": len(df.columns),
        "fields": ", ".join(map(str, df.columns)),
        "sha256_before": raw_hash_before[name],
    })

source_structure_result = pd.DataFrame(source_rows)
display(source_structure_result)

for name, df in datasets.items():
    print(f"\n== {name.upper()} SAMPLE ==")
    display(df.head(3))

print("SOURCE STRUCTURE & RAW SNAPSHOT: READY")

,dataset,physical_file,format,record_structure,records,columns,fields,sha256_before
0,orders,nusamart_orders.xlsx,XLSX,row,320,8,"order_id, customer_id, order_date, sales_chann...",bfb422563f893805b8bfca98103e437fc395495a124e6d...
1,order_items,nusamart_order_items.csv,CSV,row,763,8,"order_item_id, order_id, product_id, product_c...",4326f3690767149d781933713c19407a19b7c0a85e4d3c...
2,customers,nusamart_customers.json,JSON,object,215,8,"customer_id, customer_name, city, customer_seg...",e5d6eae9d215bbf835ddee9c87d157ec8efb6a3d35517c...
3,payments,nusamart_payments.txt,TXT,line,337,6,"payment_id, order_id, payment_date, payment_me...",47d130703fd3b6c211014bb27bd255fcc645b4a771728b...



== ORDERS SAMPLE ==


,order_id,customer_id,order_date,sales_channel,order_status,shipping_city,shipping_fee,order_total
0,ORD0001,CUS024,2026-03-25 00:00:00,Mobile App,Completed,Jakarta,20000,1450000
1,ORD0002,CUS163,2026-02-19 00:00:00,Mobile App,Processing,Yogyakarta,40000,641000
2,ORD0003,CUS202,2026-03-12 00:00:00,Mobile App,Completed,Malang,25000,482000



== ORDER_ITEMS SAMPLE ==


,order_item_id,order_id,product_id,product_category,quantity,unit_price,discount_amount,line_amount
0,ITM00001,ORD0001,PRD024,Electronics,1,1343000,0,1343000
1,ITM00002,ORD0001,PRD007,Grocery,1,65000,5000,60000
2,ITM00003,ORD0001,PRD043,Grocery,1,27000,0,27000



== CUSTOMERS SAMPLE ==


,customer_id,customer_name,city,customer_segment,join_date,email,phone,customer_status
0,CUS001,Wahyu Utami,Semarang,Regular,2025-05-28,wahyu.utami.001@example.com,081210007919,Active
1,CUS002,Nina Nugraha,Semarang,Regular,2025-04-18,nina.nugraha.002@example.com,081210015838,Active
2,CUS003,Omar Lestari,Depok,New,2025-09-01,omar.lestari.003@example.com,081210023757,Active



== PAYMENTS SAMPLE ==


,payment_id,order_id,payment_date,payment_method,payment_amount,payment_status
0,PAY0001,ORD0001,2026-03-27,QRIS,564000,Paid
1,PAY0002,ORD0001,2026-03-25,QRIS,886000,Paid
2,PAY0003,ORD0002,2026-02-21,Bank Transfer,233000,Pending


SOURCE STRUCTURE & RAW SNAPSHOT: READY


In [5]:
SOURCE_STRUCTURE = {
    "orders": {"format": "XLSX", "record_structure": "row"},
    "order_items": {"format": "CSV", "record_structure": "row"},
    "customers": {"format": "JSON", "record_structure": "object"},
    "payments": {"format": "TXT", "record_structure": "line"},
}

def file_sha256(path: Path) -> str:
    digest = sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

raw_hash_before = {name: file_sha256(path) for name, path in FILES.items()}

source_rows = []
for name, df in datasets.items():
    meta = SOURCE_STRUCTURE[name]
    source_rows.append({
        "dataset": name,
        "physical_file": FILES[name].name,
        "format": meta["format"],
        "record_structure": meta["record_structure"],
        "records": len(df),
        "columns": len(df.columns),
        "fields": ", ".join(map(str, df.columns)),
        "sha256_before": raw_hash_before[name],
    })

source_structure_result = pd.DataFrame(source_rows)
display(source_structure_result)

for name, df in datasets.items():
    print(f"\n== {name.upper()} SAMPLE ==")
    display(df.head(3))

print("SOURCE STRUCTURE & RAW SNAPSHOT: READY")

,dataset,physical_file,format,record_structure,records,columns,fields,sha256_before
0,orders,nusamart_orders.xlsx,XLSX,row,320,8,"order_id, customer_id, order_date, sales_chann...",bfb422563f893805b8bfca98103e437fc395495a124e6d...
1,order_items,nusamart_order_items.csv,CSV,row,763,8,"order_item_id, order_id, product_id, product_c...",4326f3690767149d781933713c19407a19b7c0a85e4d3c...
2,customers,nusamart_customers.json,JSON,object,215,8,"customer_id, customer_name, city, customer_seg...",e5d6eae9d215bbf835ddee9c87d157ec8efb6a3d35517c...
3,payments,nusamart_payments.txt,TXT,line,337,6,"payment_id, order_id, payment_date, payment_me...",47d130703fd3b6c211014bb27bd255fcc645b4a771728b...



== ORDERS SAMPLE ==


,order_id,customer_id,order_date,sales_channel,order_status,shipping_city,shipping_fee,order_total
0,ORD0001,CUS024,2026-03-25 00:00:00,Mobile App,Completed,Jakarta,20000,1450000
1,ORD0002,CUS163,2026-02-19 00:00:00,Mobile App,Processing,Yogyakarta,40000,641000
2,ORD0003,CUS202,2026-03-12 00:00:00,Mobile App,Completed,Malang,25000,482000



== ORDER_ITEMS SAMPLE ==


,order_item_id,order_id,product_id,product_category,quantity,unit_price,discount_amount,line_amount
0,ITM00001,ORD0001,PRD024,Electronics,1,1343000,0,1343000
1,ITM00002,ORD0001,PRD007,Grocery,1,65000,5000,60000
2,ITM00003,ORD0001,PRD043,Grocery,1,27000,0,27000



== CUSTOMERS SAMPLE ==


,customer_id,customer_name,city,customer_segment,join_date,email,phone,customer_status
0,CUS001,Wahyu Utami,Semarang,Regular,2025-05-28,wahyu.utami.001@example.com,081210007919,Active
1,CUS002,Nina Nugraha,Semarang,Regular,2025-04-18,nina.nugraha.002@example.com,081210015838,Active
2,CUS003,Omar Lestari,Depok,New,2025-09-01,omar.lestari.003@example.com,081210023757,Active



== PAYMENTS SAMPLE ==


,payment_id,order_id,payment_date,payment_method,payment_amount,payment_status
0,PAY0001,ORD0001,2026-03-27,QRIS,564000,Paid
1,PAY0002,ORD0001,2026-03-25,QRIS,886000,Paid
2,PAY0003,ORD0002,2026-02-21,Bank Transfer,233000,Pending


SOURCE STRUCTURE & RAW SNAPSHOT: READY


## 3. Grain & Candidate Key

Isi `02_Grain_Key` terlebih dahulu.

Peserta hanya mengisi:
- grain statement;
- candidate key.

Technical key checks sudah tersedia.

### Tabel Orders

In [13]:
import duckdb

# Menjalankan query SQL pengecekan candidate key
query = """
SELECT
    COUNT(*) AS total_orders,
    COUNT(DISTINCT order_id) AS unique_orders,
    SUM(CASE WHEN order_id IS NULL OR TRIM(order_id) = '' THEN 1 ELSE 0 END) AS missing_order_id
FROM df_orders;
"""
display(duckdb.query(query).to_df())


,total_orders,unique_orders,missing_order_id
0,320,320,0.0


### Tabel Order Items

In [21]:
import pandas as pd
import duckdb

# 1. Kita load ulang datanya ke variabel lokal agar PASTI ada di memori sel ini
df_order_items_lokal = pd.read_csv(FILES["order_items"])

# 2. Daftarkan ke DuckDB dengan nama tabel 'tabel_items'
duckdb.register("tabel_items", df_order_items_lokal)

# 3. Jalankan query SQL menggunakan nama tabel yang sudah dikunci pasti ada
query = """
SELECT
    COUNT(*) AS total_order_items,
    COUNT(DISTINCT order_item_id) AS unique_order_items,
    SUM(CASE WHEN order_item_id IS NULL OR TRIM(CAST(order_item_id AS VARCHAR)) = '' THEN 1 ELSE 0 END) AS missing_order_item_id
FROM tabel_items;
"""

display(duckdb.query(query).to_df())


,total_order_items,unique_order_items,missing_order_item_id
0,763,760,0.0


In [22]:
# Kueri untuk mencari ID mana saja yang muncul lebih dari 1 kali
query_cek_duplikat = """
SELECT 
    order_item_id, 
    COUNT(*) as jumlah_kemunculan
FROM tabel_items
GROUP BY order_item_id
HAVING COUNT(*) > 1;
"""
display(duckdb.query(query_cek_duplikat).to_df())


,order_item_id,jumlah_kemunculan
0,ITM00130,2
1,ITM00215,2
2,ITM00464,2


### Tabel Customers

In [24]:
import pandas as pd
import duckdb

# 1. Membaca file JSON format standar (tanpa parameter lines=True)
df_customers_lokal = pd.read_json(FILES["customers"])

# 2. Daftarkan ke DuckDB sebagai tabel lokal
duckdb.register("tabel_customers", df_customers_lokal)

# 3. Jalankan query SQL pengecekan candidate key
query_customers = """
SELECT
    COUNT(*) AS total_records,
    COUNT(DISTINCT customer_id) AS key_distinct_count,
    SUM(CASE WHEN customer_id IS NULL OR TRIM(CAST(customer_id AS VARCHAR)) = '' THEN 1 ELSE 0 END) AS key_missing_count
FROM tabel_customers;
"""

print("=== HASIL DATASET: CUSTOMERS ===")
display(duckdb.query(query_customers).to_df())


=== HASIL DATASET: CUSTOMERS ===


,total_records,key_distinct_count,key_missing_count
0,215,215,0.0


### Tabel Payments

In [28]:
import pandas as pd
import duckdb

# 1. Membaca data tanpa header asli, lalu kita namai sendiri kolomnya secara rapi
kolom_baru = ["PAYMENT_ID", "ORDER_ID", "DATE", "METHOD", "AMOUNT"]
df_payments_lokal = pd.read_csv(FILES["payments"], sep="|", header=None, names=kolom_baru, skipinitialspace=True)

# 2. Daftarkan ke DuckDB sebagai tabel lokal
duckdb.register("tabel_payments", df_payments_lokal)

# 3. Jalankan query SQL pengecekan candidate key
query_payments = """
SELECT
    COUNT(*) AS total_records,
    COUNT(DISTINCT PAYMENT_ID) AS key_distinct_count,
    SUM(CASE WHEN PAYMENT_ID IS NULL OR TRIM(CAST(PAYMENT_ID AS VARCHAR)) = '' THEN 1 ELSE 0 END) AS key_missing_count
FROM tabel_payments;
"""

print("=== HASIL DATASET: PAYMENTS ===")
display(duckdb.query(query_payments).to_df())


=== HASIL DATASET: PAYMENTS ===


,total_records,key_distinct_count,key_missing_count
0,337,290,0.0


In [30]:
# Menghitung baris yang isinya sama persis dari ujung ke ujung
jumlah_duplikat_total = df_payments_lokal.duplicated().sum()

print(f"Jumlah baris yang duplikat total (identik): {jumlah_duplikat_total} baris")
print(f"Jumlah ID yang berulang tetapi beda isi data : {47 - jumlah_duplikat_total} baris")


Jumlah baris yang duplikat total (identik): 2 baris
Jumlah ID yang berulang tetapi beda isi data : 45 baris


In [35]:
import duckdb

# Query untuk menampilkan ID yang kembar beserta jumlah kemunculannya persis seperti di gambar
query_duplikat_payments = """
SELECT 
    PAYMENT_ID, 
    COUNT(*) AS jumlah_kemunculan
FROM tabel_payments
GROUP BY PAYMENT_ID
HAVING COUNT(*) > 1
ORDER BY jumlah_kemunculan DESC;
"""

print("=== DAFTAR PAYMENT_ID YANG DUPLIKAT ===")
display(duckdb.query(query_duplikat_payments).to_df())


=== DAFTAR PAYMENT_ID YANG DUPLIKAT ===


,PAYMENT_ID,jumlah_kemunculan
0,ORDER_ID=ORD0014,2
1,ORDER_ID=ORD0192,2
2,ORDER_ID=ORD0022,2
3,ORDER_ID=ORD0283,2
4,ORDER_ID=ORD0284,2
5,ORDER_ID=ORD0320,2
6,ORDER_ID=ORD0155,2
7,ORDER_ID=ORD0157,2
8,ORDER_ID=ORD0163,2
9,ORDER_ID=ORD0247,2


In [6]:
GRAIN_KEY_PLAN = {
    "orders": {
        "grain_statement": "1 record (row) = 1 order",
        "candidate_key": "order_id",
    },
    "order_items": {
        "grain_statement": "1 record (row) = 1 item dalam 1 order",
        "candidate_key": "order_item_id",
    },
    "customers": {
        "grain_statement": "1 record (object) = 1 customer",
        "candidate_key": "customer_id",
    },
    "payments": {
        "grain_statement": "1 record (line) = 1 transaksi pembayaran",
        "candidate_key": "payment_id",
    },
}

def is_todo(value) -> bool:
    return str(value).strip().upper() in {"", "TODO", "<TODO>"}

def missing_or_blank_mask(series: pd.Series) -> pd.Series:
    text = series.astype("string")
    return text.isna() | text.str.strip().eq("")

def check_candidate_key(dataset_name: str, candidate_key: str) -> dict:
    df = datasets[dataset_name]
    if candidate_key not in df.columns:
        raise KeyError(
            f"{dataset_name}: field '{candidate_key}' tidak ditemukan. "
            f"Fields: {list(df.columns)}"
        )

    s = df[candidate_key]
    missing_mask = missing_or_blank_mask(s)
    repeated = (
        df.loc[~missing_mask, [candidate_key]]
        .value_counts(dropna=False)
        .reset_index(name="n")
    )
    repeated = repeated[repeated["n"] > 1].head(5)

    return {
        "dataset": dataset_name,
        "grain_statement": GRAIN_KEY_PLAN[dataset_name]["grain_statement"],
        "candidate_key": candidate_key,
        "total_records": len(df),
        "distinct_key": int(s[~missing_mask].nunique(dropna=True)),
        "missing_or_blank_key": int(missing_mask.sum()),
        "unique_on_raw": bool((~missing_mask).sum() == s[~missing_mask].nunique(dropna=True)),
        "repeated_key_sample": repeated.to_dict("records"),
    }

grain_key_issues = []
grain_key_results = []

for dataset_name, cfg in GRAIN_KEY_PLAN.items():
    grain = cfg.get("grain_statement")
    key = cfg.get("candidate_key")

    if is_todo(grain):
        grain_key_issues.append(f"{dataset_name}: grain_statement belum diisi.")
        continue
    if is_todo(key):
        grain_key_issues.append(f"{dataset_name}: candidate_key belum diisi.")
        continue
    if key not in datasets[dataset_name].columns:
        grain_key_issues.append(
            f"{dataset_name}: candidate_key '{key}' tidak ada di dataset."
        )
        continue

    grain_key_results.append(check_candidate_key(dataset_name, key))

grain_key_gate_pass = len(grain_key_issues) == 0 and len(grain_key_results) == 4

if grain_key_results:
    display(pd.DataFrame(grain_key_results))

if grain_key_gate_pass:
    print("GRAIN / KEY GATE: PASS")
else:
    print("GRAIN / KEY GATE: NEEDS INPUT / REVISION")
    for issue in grain_key_issues:
        print(" -", issue)

print("\nRepeated key adalah evidence untuk review, bukan otomatis exact duplicate.")

,dataset,grain_statement,candidate_key,total_records,distinct_key,missing_or_blank_key,unique_on_raw,repeated_key_sample
0,orders,1 record (row) = 1 order,order_id,320,320,0,True,[]
1,order_items,1 record (row) = 1 item dalam 1 order,order_item_id,763,760,0,False,"[{'order_item_id': 'ITM00130', 'n': 2}, {'orde..."
2,customers,1 record (object) = 1 customer,customer_id,215,215,0,True,[]
3,payments,1 record (line) = 1 transaksi pembayaran,payment_id,337,335,0,False,"[{'payment_id': 'PAY0095', 'n': 2}, {'payment_..."


GRAIN / KEY GATE: PASS

Repeated key adalah evidence untuk review, bukan otomatis exact duplicate.


## 4. Data Quality Criteria — BEFORE Profiling

Isi `03_DQ_Criteria` **sebelum** menjalankan profiling utama.

Criteria adalah standar pemeriksaan, bukan finding dan bukan cleaning rule.

Setelah Working Paper memiliki criteria yang cukup untuk:
- completeness,
- uniqueness,
- validity,
- consistency,
- chronology,
- business logic / range,

ubah `DQ_CRITERIA_CONFIRMED` menjadi `True`.

In [7]:
DQ_CRITERIA_CONFIRMED = True

if DQ_CRITERIA_CONFIRMED:
    print("DQ CRITERIA GATE: PASS")
else:
    print(
        "DQ CRITERIA GATE: NEEDS INPUT — isi 03_DQ_Criteria lebih dulu, "
        "lalu ubah DQ_CRITERIA_CONFIRMED = True."
    )

DQ CRITERIA GATE: PASS


## 5. Manual Profiling — Guided Check Catalog

Minimum kelas: **≥10 checks**.

Starter menyediakan catalog 15 checks yang mengikuti Working Paper FILLED/dry-run lama, tetapi **tidak mengisi hasilnya**.

Peserta memilih check yang akan dijalankan melalui `SELECTED_PROFILING_CHECK_IDS`.

Untuk MP-15, pilih satu categorical field Orders yang relevan pada `ORDERS_CATEGORY_FIELD`.

In [8]:
ORDERS_CATEGORY_FIELD = ""  # isi hanya jika memilih MP-15

PROFILING_CHECK_CATALOG = pd.DataFrame([
    ["MP-01","orders","all rows","Record Count","Berapa logical records Orders yang terbaca?"],
    ["MP-02","order_items","all rows","Record Count","Berapa logical records Order Items yang terbaca?"],
    ["MP-03","customers","all records","Record Count","Berapa logical customer records yang terbaca?"],
    ["MP-04","payments","all lines","Record Count","Berapa logical payment records yang terbaca?"],
    ["MP-05","order_items","candidate key","Repeated Key","Apakah candidate key Order Items muncul lebih dari sekali?"],
    ["MP-06","payments","candidate key","Repeated Key","Apakah candidate key Payments muncul lebih dari sekali?"],
    ["MP-07","customers","customer_segment","Missing / Blank","Apakah customer_segment memiliki missing/blank?"],
    ["MP-08","customers","customer_segment","Cardinality / Category","Bagaimana distribusi customer_segment?"],
    ["MP-09","order_items","full row","Exact Duplicate","Apakah ada full-row exact duplicate pada Order Items?"],
    ["MP-10","payments","full row","Exact Duplicate","Apakah ada full-row exact duplicate pada Payments?"],
    ["MP-11","orders","order_date","Date / Representation","Apakah order_date memiliki parseability/representation issue?"],
    ["MP-12","payments","payment_date","Date / Representation","Apakah payment_date memiliki parseability/representation issue?"],
    ["MP-13","payments","payment_amount","Numeric Validity","Apakah payment_amount memiliki representation/validity issue?"],
    ["MP-14","order_items","quantity","Range / Suspicious Value","Bagaimana min/max/avg quantity dan record maksimum?"],
    ["MP-15","orders","participant-selected field","Cardinality / Category","Apakah ada category variation pada field kategori Orders yang dipilih?"],
], columns=["check_id","dataset","field_or_area","check_type","question"])

display(PROFILING_CHECK_CATALOG)

SELECTED_PROFILING_CHECK_IDS = [   "MP-01",
    "MP-05",
    "MP-07",
    "MP-08",
    "MP-09",
    "MP-11",
    "MP-12",
    "MP-13",
    "MP-14",
    "MP-03",
    "MP-01"]  # pilih minimal 10, contoh: ["MP-01", "MP-02", ...]

,check_id,dataset,field_or_area,check_type,question
0,MP-01,orders,all rows,Record Count,Berapa logical records Orders yang terbaca?
1,MP-02,order_items,all rows,Record Count,Berapa logical records Order Items yang terbaca?
2,MP-03,customers,all records,Record Count,Berapa logical customer records yang terbaca?
3,MP-04,payments,all lines,Record Count,Berapa logical payment records yang terbaca?
4,MP-05,order_items,candidate key,Repeated Key,Apakah candidate key Order Items muncul lebih ...
5,MP-06,payments,candidate key,Repeated Key,Apakah candidate key Payments muncul lebih dar...
6,MP-07,customers,customer_segment,Missing / Blank,Apakah customer_segment memiliki missing/blank?
7,MP-08,customers,customer_segment,Cardinality / Category,Bagaimana distribusi customer_segment?
8,MP-09,order_items,full row,Exact Duplicate,Apakah ada full-row exact duplicate pada Order...
9,MP-10,payments,full row,Exact Duplicate,Apakah ada full-row exact duplicate pada Payme...


In [9]:
def parse_dates_mixed(series: pd.Series) -> pd.Series:
    try:
        return pd.to_datetime(series, errors="coerce", format="mixed")
    except (TypeError, ValueError):
        return pd.to_datetime(series, errors="coerce")

def date_representation_counts(series: pd.Series) -> dict:
    def signature(value):
        if pd.isna(value):
            return "<MISSING>"
        if isinstance(value, (pd.Timestamp,)):
            return "datetime-like"

        s = str(value).strip()
        patterns = [
            (r"^\d{4}-\d{2}-\d{2}$", "YYYY-MM-DD"),
            (r"^\d{2}/\d{2}/\d{4}$", "DD/MM/YYYY or MM/DD/YYYY"),
            (r"^\d{2}-\d{2}-\d{4}$", "DD-MM-YYYY or MM-DD-YYYY"),
            (r"^\d{4}/\d{2}/\d{2}$", "YYYY/MM/DD"),
        ]
        for pattern, label in patterns:
            if re.match(pattern, s):
                return label
        return "OTHER"

    return series.map(signature).value_counts(dropna=False).to_dict()

def profile_record_count(dataset_name: str) -> dict:
    n = len(datasets[dataset_name])
    return {
        "result_value": n,
        "manual_result": f"{n} records",
        "evidence_preview": f"COUNT(*) = {n}",
    }

def profile_repeated_key(dataset_name: str) -> dict:
    key = GRAIN_KEY_PLAN[dataset_name]["candidate_key"]
    if is_todo(key):
        raise ValueError(f"{dataset_name}: candidate key belum diisi pada GRAIN_KEY_PLAN.")

    df = datasets[dataset_name]
    missing = missing_or_blank_mask(df[key])
    counts = df.loc[~missing, key].value_counts(dropna=False)
    repeated = counts[counts > 1]
    repeated_records = int(repeated.sum())
    repeated_distinct_keys = int(len(repeated))
    sample = repeated.head(5).to_dict()

    return {
        "result_value": repeated_distinct_keys,
        "manual_result": (
            f"repeated_distinct_keys={repeated_distinct_keys}; "
            f"records_under_repeated_keys={repeated_records}"
        ),
        "evidence_preview": str(sample),
    }

def profile_missing_blank(dataset_name: str, field: str) -> dict:
    df = datasets[dataset_name]
    mask = missing_or_blank_mask(df[field])
    n = int(mask.sum())
    pct = 0.0 if len(df) == 0 else n / len(df) * 100
    sample_index = df.index[mask].tolist()[:5]
    return {
        "result_value": n,
        "manual_result": f"missing_or_blank={n}; pct={pct:.2f}%",
        "evidence_preview": f"sample_row_index={sample_index}",
    }

def profile_category(dataset_name: str, field: str) -> dict:
    df = datasets[dataset_name].copy()
    if field not in df.columns:
        raise KeyError(f"{dataset_name}: field '{field}' tidak ditemukan.")
    values = df[field].astype("string").fillna("<MISSING>")
    freq = values.value_counts(dropna=False)
    top = freq.head(10).to_dict()
    return {
        "result_value": int(values.nunique(dropna=False)),
        "manual_result": f"distinct_categories={int(values.nunique(dropna=False))}",
        "evidence_preview": str(top),
    }

def profile_exact_duplicates(dataset_name: str) -> dict:
    df = datasets[dataset_name]
    duplicate_extras = int(df.duplicated(keep="first").sum())
    repeated_rows = df[df.duplicated(keep=False)].head(6)
    return {
        "result_value": duplicate_extras,
        "manual_result": f"exact_duplicate_extras={duplicate_extras}",
        "evidence_preview": repeated_rows.to_dict("records"),
    }

def profile_date(dataset_name: str, field: str) -> dict:
    df = datasets[dataset_name]
    raw = df[field]
    parsed = parse_dates_mixed(raw)
    unresolved = int(parsed.isna().sum())
    rep = date_representation_counts(raw)
    min_date = None if parsed.dropna().empty else parsed.min()
    max_date = None if parsed.dropna().empty else parsed.max()

    return {
        "result_value": unresolved,
        "manual_result": (
            f"records={len(df)}; unresolved_dates={unresolved}; "
            f"min_parsed={min_date}; max_parsed={max_date}"
        ),
        "evidence_preview": f"representation_counts={rep}",
    }

def profile_numeric(dataset_name: str, field: str) -> dict:
    df = datasets[dataset_name]
    raw = df[field]
    parsed = pd.to_numeric(raw, errors="coerce")
    raw_missing = missing_or_blank_mask(raw)
    invalid_nonblank = int((parsed.isna() & ~raw_missing).sum())
    valid = parsed.dropna()

    min_v = None if valid.empty else float(valid.min())
    max_v = None if valid.empty else float(valid.max())
    avg_v = None if valid.empty else float(valid.mean())
    nonpositive = int((valid <= 0).sum()) if not valid.empty else 0

    max_rows = []
    if max_v is not None:
        max_rows = df.loc[parsed.eq(max_v)].head(5).to_dict("records")

    return {
        "result_value": max_v,
        "manual_result": (
            f"invalid_nonblank={invalid_nonblank}; min={min_v}; "
            f"max={max_v}; avg={avg_v}; nonpositive={nonpositive}"
        ),
        "evidence_preview": f"max_value_rows={max_rows}",
    }

def run_profile_check(check_id: str) -> dict:
    if check_id == "MP-01":
        payload = profile_record_count("orders")
    elif check_id == "MP-02":
        payload = profile_record_count("order_items")
    elif check_id == "MP-03":
        payload = profile_record_count("customers")
    elif check_id == "MP-04":
        payload = profile_record_count("payments")
    elif check_id == "MP-05":
        payload = profile_repeated_key("order_items")
    elif check_id == "MP-06":
        payload = profile_repeated_key("payments")
    elif check_id == "MP-07":
        payload = profile_missing_blank("customers", "customer_segment")
    elif check_id == "MP-08":
        payload = profile_category("customers", "customer_segment")
    elif check_id == "MP-09":
        payload = profile_exact_duplicates("order_items")
    elif check_id == "MP-10":
        payload = profile_exact_duplicates("payments")
    elif check_id == "MP-11":
        payload = profile_date("orders", "order_date")
    elif check_id == "MP-12":
        payload = profile_date("payments", "payment_date")
    elif check_id == "MP-13":
        payload = profile_numeric("payments", "payment_amount")
    elif check_id == "MP-14":
        payload = profile_numeric("order_items", "quantity")
    elif check_id == "MP-15":
        if not ORDERS_CATEGORY_FIELD.strip():
            raise ValueError("Isi ORDERS_CATEGORY_FIELD sebelum menjalankan MP-15.")
        payload = profile_category("orders", ORDERS_CATEGORY_FIELD.strip())
    else:
        raise KeyError(f"Unknown profiling check ID: {check_id}")

    catalog_row = PROFILING_CHECK_CATALOG.loc[
        PROFILING_CHECK_CATALOG["check_id"].eq(check_id)
    ].iloc[0]

    return {
        "check_id": check_id,
        "dataset": catalog_row["dataset"],
        "field_or_area": (
            ORDERS_CATEGORY_FIELD.strip()
            if check_id == "MP-15"
            else catalog_row["field_or_area"]
        ),
        "check_type": catalog_row["check_type"],
        "question": catalog_row["question"],
        **payload,
        "status": "PASS",
    }

selected_check_ids = [str(x).strip().upper() for x in SELECTED_PROFILING_CHECK_IDS]
selected_check_ids = list(dict.fromkeys(selected_check_ids))

known_ids = set(PROFILING_CHECK_CATALOG["check_id"])
unknown_ids = [x for x in selected_check_ids if x not in known_ids]

manual_profiling_results = None
manual_profiling_gate_pass = False
manual_profiling_issues = []

if not grain_key_gate_pass:
    manual_profiling_issues.append("GRAIN / KEY GATE belum PASS.")
if not DQ_CRITERIA_CONFIRMED:
    manual_profiling_issues.append("DQ CRITERIA GATE belum PASS.")
if len(selected_check_ids) < 10:
    manual_profiling_issues.append(
        f"Minimal 10 checks diperlukan; saat ini dipilih {len(selected_check_ids)}."
    )
if unknown_ids:
    manual_profiling_issues.append(f"Unknown check IDs: {unknown_ids}")
if "MP-15" in selected_check_ids and not ORDERS_CATEGORY_FIELD.strip():
    manual_profiling_issues.append("MP-15 dipilih tetapi ORDERS_CATEGORY_FIELD belum diisi.")

if manual_profiling_issues:
    print("MANUAL PROFILING GATE: NEEDS INPUT / REVISION")
    for issue in manual_profiling_issues:
        print(" -", issue)
else:
    rows = []
    for check_id in selected_check_ids:
        rows.append(run_profile_check(check_id))

    manual_profiling_results = pd.DataFrame(rows)
    manual_profiling_gate_pass = (
        len(manual_profiling_results) >= 10
        and (manual_profiling_results["status"] == "PASS").all()
    )

    display(
        manual_profiling_results[
            [
                "check_id","dataset","field_or_area","check_type",
                "question","manual_result","evidence_preview"
            ]
        ]
    )
    print("MANUAL PROFILING GATE:", "PASS" if manual_profiling_gate_pass else "FAIL")

print("\nSalin evidence yang relevan ke 04_Manual_Profiling. Jangan cleaning.")

,check_id,dataset,field_or_area,check_type,question,manual_result,evidence_preview
0,MP-01,orders,all rows,Record Count,Berapa logical records Orders yang terbaca?,320 records,COUNT(*) = 320
1,MP-05,order_items,candidate key,Repeated Key,Apakah candidate key Order Items muncul lebih ...,repeated_distinct_keys=3; records_under_repeat...,"{'ITM00130': 2, 'ITM00215': 2, 'ITM00464': 2}"
2,MP-07,customers,customer_segment,Missing / Blank,Apakah customer_segment memiliki missing/blank?,missing_or_blank=4; pct=1.86%,"sample_row_index=[70, 107, 117, 173]"
3,MP-08,customers,customer_segment,Cardinality / Category,Bagaimana distribusi customer_segment?,distinct_categories=8,"{'Regular': 111, 'New': 52, 'VIP': 42, '<MISSI..."
4,MP-09,order_items,full row,Exact Duplicate,Apakah ada full-row exact duplicate pada Order...,exact_duplicate_extras=3,"[{'order_item_id': 'ITM00130', 'order_id': 'OR..."
5,MP-11,orders,order_date,Date / Representation,Apakah order_date memiliki parseability/repres...,records=320; unresolved_dates=0; min_parsed=20...,"representation_counts={'OTHER': 316, 'DD/MM/YY..."
6,MP-12,payments,payment_date,Date / Representation,Apakah payment_date memiliki parseability/repr...,records=337; unresolved_dates=0; min_parsed=20...,"representation_counts={'YYYY-MM-DD': 334, 'DD-..."
7,MP-13,payments,payment_amount,Numeric Validity,Apakah payment_amount memiliki representation/...,invalid_nonblank=3; min=-344000.0; max=4176000...,"max_value_rows=[{'payment_id': 'PAY0247', 'ord..."
8,MP-14,order_items,quantity,Range / Suspicious Value,Bagaimana min/max/avg quantity dan record maks...,invalid_nonblank=0; min=1.0; max=30.0; avg=1.4...,"max_value_rows=[{'order_item_id': 'ITM00676', ..."
9,MP-03,customers,all records,Record Count,Berapa logical customer records yang terbaca?,215 records,COUNT(*) = 215


MANUAL PROFILING GATE: PASS

Salin evidence yang relevan ke 04_Manual_Profiling. Jangan cleaning.


### Cek Manual orders - Record Count

In [17]:
import pandas as pd
import duckdb

orders = pd.read_excel("data/raw/nusamart/nusamart_orders.xlsx")

result = duckdb.sql("""
SELECT COUNT(*) AS logical_record_count
FROM orders;
""").df()

display(result)

,logical_record_count
0,320


### Cek Manual Order_items - Repeated Key

In [21]:
import pandas as pd
import duckdb

order_items = pd.read_csv("data/raw/nusamart/nusamart_order_items.csv")

result = duckdb.sql("""
SELECT
    COUNT(*) AS repeated_distinct_keys
FROM (
    SELECT order_item_id
    FROM order_items
    GROUP BY order_item_id
    HAVING COUNT(*) > 1
) AS duplicates;
""").df()

display(result)

,repeated_distinct_keys
0,3


In [30]:
customers = pd.read_json("data/raw/nusamart/nusamart_customers.json")

customers = pd.json_normalize(
    customers.to_dict(orient="records")
)

print(customers.columns.tolist())

['customer_id', 'customer_status', 'profile.customer_name', 'profile.city', 'profile.customer_segment', 'profile.join_date', 'contact.email', 'contact.phone']


### Cek Manual Customers_segment - Missing Blank

In [31]:
import pandas as pd
import duckdb

result = duckdb.sql("""
SELECT COUNT(*) AS missing_or_blank_count
FROM customers
WHERE "profile.customer_segment" IS NULL
   OR TRIM("profile.customer_segment") = '';
""").df()

display(result)


,missing_or_blank_count
0,4


### Cek Manual Customers_segment - Cardinality/Category

In [33]:
import pandas as pd
import duckdb

result = duckdb.sql("""
SELECT "profile.customer_segment", 
COUNT(*) AS record_count FROM customers 
GROUP BY "profile.customer_segment" ORDER BY record_count DESC;
""").df()

display(result)

,profile.customer_segment,record_count
0,Regular,111
1,New,52
2,VIP,42
3,NaN,4
4,regular,3
5,REG,1
6,new customer,1
7,vip,1


### Cek Manual Order_items - Exact Duplicate

In [38]:
import pandas as pd
import duckdb

order_items = pd.read_csv("data/raw/nusamart/nusamart_order_items.csv")

result = duckdb.sql("""
SELECT COUNT(*) AS duplicate_rows
FROM (
    SELECT *,
           COUNT(*) OVER (PARTITION BY COLUMNS(*)) AS cnt
    FROM order_items
)
WHERE cnt > 1;
""").df()

display(result)

,duplicate_rows
0,6


### Cek Manual Payment_date - Representation

### Cek Manual Orders_date - Representation

In [41]:
result = profile_date("orders", "order_date")
display(result)

{'result_value': 0,
 'manual_result': 'records=320; unresolved_dates=0; min_parsed=2026-01-01 00:00:00; max_parsed=2026-04-02 00:00:00',
 'evidence_preview': "representation_counts={'OTHER': 316, 'DD/MM/YYYY or MM/DD/YYYY': 2, 'YYYY/MM/DD': 1, 'DD-MM-YYYY or MM-DD-YYYY': 1}"}

Semua tanggal dapat diparse, tetapi terdapat beberapa variasi format tanggal

In [40]:
import pandas as pd
import duckdb

orders = pd.read_excel("data/raw/nusamart/nusamart_orders.xlsx")

result = duckdb.sql("""
SELECT
    COUNT(*) AS total_records,
    COUNT(*) - COUNT(TRY_CAST(order_date AS DATE)) AS unparseable_dates,
    COUNT(DISTINCT order_date) AS distinct_date_values
FROM orders;
""").df()

display(result)

,total_records,unparseable_dates,distinct_date_values
0,320,3,94


### Cek Manual Order_items - Quantity (Range/Dist)

In [39]:
import pandas as pd
import duckdb

order_items = pd.read_csv("data/raw/nusamart/nusamart_order_items.csv")

result = duckdb.sql("""
SELECT MIN(quantity) AS min_quantity,
 MAX(quantity) AS max_quantity, AVG(quantity) AS avg_quantity, 
 COUNT(*) AS record_count FROM order_items;
""").df()

display(result)

,min_quantity,max_quantity,avg_quantity,record_count
0,1,30,1.410223,763


In [42]:
result = duckdb.sql("""
SELECT *
FROM order_items
WHERE quantity = (
    SELECT MAX(quantity)
    FROM order_items
);
""").df()

display(result)

,order_item_id,order_id,product_id,product_category,quantity,unit_price,discount_amount,line_amount
0,ITM00676,ORD0288,PRD025,Grocery,30,19000,17000,553000


## 6. Antigravity Profiling Handoff

Setelah:
- `01_Source_Structure` terisi,
- `02_Grain_Key` terisi,
- `03_DQ_Criteria` selesai,
- `04_Manual_Profiling` memiliki ≥10 checks,

gunakan **S2 NusaMart — Antigravity Profiling Brief**.

Output Antigravity wajib:

`outputs/sesi_02/profiling_report.md`

Notebook tidak membuat report tersebut; notebook hanya mengecek apakah file sudah tersedia untuk tahap verification.

In [11]:
profiling_report_exists = PROFILING_REPORT.exists()

if profiling_report_exists:
    report_text = PROFILING_REPORT.read_text(encoding="utf-8", errors="replace")
    print("ANTIGRAVITY REPORT CHECK: PASS")
    print("path:", PROFILING_REPORT)
    print("chars:", len(report_text))
    print("\n--- preview ---")
    print(report_text[:1800])
else:
    report_text = ""
    print("ANTIGRAVITY REPORT CHECK: NOT READY")
    print("Expected:", PROFILING_REPORT)
    print("Jalankan Antigravity menggunakan Brief S2 setelah manual baseline selesai.")

ANTIGRAVITY REPORT CHECK: PASS
path: C:\Bootcamp\Bootcamp_Data_Analyst\outputs\sesi_02\profiling_report.md
chars: 45040

--- preview ---
# NusaMart Data Profiling Report — Sesi 02

**Scope Dataset**: `orders`, `order_items`, `customers`, `payments`  
**Execution Type**: Automated Profiling Baseline (Antigravity)  
**Tujuan**: Mengidentifikasi metadata struktural, keunikan candidate key, kelengkapan data, variasi representasi kategori/tanggal/angka, serta candidate issues untuk diverifikasi manual oleh peserta.  
**Prinsip Kerja**: Detect → Measure → Verify (Tanpa data cleaning, tanpa standardisasi sepihak, tanpa relational join kompleks).

---

## 1. Executive Profiling Summary

Profiling dilakukan terhadap 4 logical datasets NusaMart yang telah di-load ke dalam memori dari berbagai format sumber raw (`.xlsx`, `.csv`, `.json`, `.txt`). Keempat dataset ini merepresentasikan entitas bisnis e-commerce utama: pesanan (*orders*), rincian produk pesanan (*order items*), master pelanggan (*cu

## 7. Manual Verification — Minimal 5 Claims

Ambil candidate claims dari `profiling_report.md`.

Untuk setiap claim:
1. pilih `manual_check_id` yang relevan;
2. Starter menjalankan kembali check manual;
3. peserta membandingkan Agent Result vs Manual Result;
4. peserta menetapkan `MATCH / MISMATCH / NEEDS_REVIEW`;
5. peserta menetapkan classification S2:
   - `DATA_QUALITY_ISSUE`
   - `VALID_BUT_UNUSUAL`
   - `AMBIGUOUS`
   - `NO_ISSUE`

Technical re-check tersedia; **judgment tetap peserta**.

In [ ]:
VERIFICATION_PLAN = [
    {
        "finding_id": "VF-02",
        "claim_from_report": "customer_segment memiliki 4 missing/blank values",
        "dataset_source": "customers",
        "agent_result": "memiliki missing/null values sebanyak 4 record (1.86%).",
        "manual_check_id": "SELECT COUNT(*) AS missing_or_blank_count FROM customers WHERE customer_segment IS NULL OR TRIM(customer_segment) = '';",
        "verification_status": "MATCH",
        "finding_classification": "DATA_QUALITY_ISSUE",
        "evidence_note": "Manual check mengonfirmasi 3 repeated candidate keys.",
        "notes": "",
    },
    {
        "finding_id": "VF-03",
        "claim_from_report": "order_items memiliki 3 exact duplicate extras",
        "dataset_source": "order_items",
        "agent_result": "exact_duplicate_extras=3",
        "manual_check_id": "SELECT *, COUNT(*) AS duplicate_count FROM order_items GROUP BY order_id, product_id, quantity HAVING COUNT(*) > 1 ORDER BY duplicate_count DESC;",
        "verification_status": "MATCH",
        "finding_classification": "DATA_QUALITY_ISSUE",
        "evidence_note": "Manual check mengonfirmasi 3 exact duplicate extras.",
        "notes": "",
    },
    {
        "finding_id": "VF-04",
        "claim_from_report": "order_id memiliki 3 exact duplicate extras",
        "dataset_source": "orders",
        "agent_result": "exact_duplicate_extras=3",
        "manual_check_id": "SELECT order_date, COUNT(*) AS record_count FROM orders WHERE order_date IS NULL OR TRIM(CAST(order_date AS VARCHAR)) = '' GROUP BY order_date;",
        "verification_status": "MATCH",
        "finding_classification": "DATA_QUALITY_ISSUE",
        "evidence_note": "Manual check mengonfirmasi 3 exact duplicate extras.",
        "notes": "",
    },
    {
        "finding_id": "VF-05",
        "claim_from_report": "payment_amount memiliki 3 invalid nonblank values",
        "dataset_source": "payments",
        "agent_result": "invalid_nonblank=3",
        "manual_check_id": "MP-13",
        "verification_status": "MATCH",
        "finding_classification": "DATA_QUALITY_ISSUE",
        "evidence_note": "Manual check mengonfirmasi adanya 3 invalid nonblank values.",
        "notes": "",
    },
    {
        "finding_id": "VF-06",
        "claim_from_report": "payment_date memiliki 0 unresolved dates",
        "dataset_source": "payments",
        "agent_result": "unresolved_dates=0",
        "manual_check_id": "MP-12",
        "verification_status": "MATCH",
        "finding_classification": "NO_ISSUE",
        "evidence_note": "Manual check menghasilkan unresolved_dates=0.",
        "notes": "",
    },
]

ALLOWED_VERIFICATION_STATUS = {"MATCH", "MISMATCH", "NEEDS_REVIEW"}
ALLOWED_CLASSIFICATION = {
    "DATA_QUALITY_ISSUE",
    "VALID_BUT_UNUSUAL",
    "AMBIGUOUS",
    "NO_ISSUE",
}

def normalize_token(value) -> str:
    return str(value).strip().upper().replace(" ", "_")

verification_rows = []
verification_issues = []

if not profiling_report_exists:
    verification_issues.append("profiling_report.md belum tersedia.")

for row in VERIFICATION_PLAN:
    finding_id = str(row.get("finding_id", "")).strip()
    claim = str(row.get("claim_from_report", "")).strip()
    dataset_source = str(row.get("dataset_source", "")).strip()
    agent_result = str(row.get("agent_result", "")).strip()
    check_id = str(row.get("manual_check_id", "")).strip().upper()
    status = normalize_token(row.get("verification_status", ""))
    classification = normalize_token(row.get("finding_classification", ""))
    evidence_note = str(row.get("evidence_note", "")).strip()

    row_issues = []

    if not finding_id:
        row_issues.append("finding_id kosong")
    if not claim:
        row_issues.append("claim_from_report kosong")
    if not dataset_source:
        row_issues.append("dataset_source kosong")
    if not agent_result:
        row_issues.append("agent_result kosong")
    if check_id not in known_ids:
        row_issues.append(f"manual_check_id tidak valid: {check_id!r}")
    if status not in ALLOWED_VERIFICATION_STATUS:
        row_issues.append(f"verification_status tidak valid: {status!r}")
    if classification not in ALLOWED_CLASSIFICATION:
        row_issues.append(f"finding_classification tidak valid: {classification!r}")
    if not evidence_note:
        row_issues.append("evidence_note kosong")

    if row_issues:
        verification_issues.append(
            f"{finding_id or '<no id>'}: " + "; ".join(row_issues)
        )
        continue

    manual = run_profile_check(check_id)

    verification_rows.append({
        "finding_id": finding_id,
        "claim_from_report": claim,
        "dataset_source": dataset_source,
        "agent_result": agent_result,
        "manual_check_id": check_id,
        "manual_result": manual["manual_result"],
        "verification_status": status,
        "evidence": f"{check_id}: {manual['evidence_preview']} | {evidence_note}",
        "finding_classification": classification,
        "notes": str(row.get("notes", "")).strip(),
    })

manual_verification_result = pd.DataFrame(verification_rows)
manual_verification_gate_pass = (
    profiling_report_exists
    and len(verification_rows) >= 5
    and len(verification_issues) == 0
)

if manual_verification_result is not None and not manual_verification_result.empty:
    display(manual_verification_result)

if manual_verification_gate_pass:
    print("MANUAL VERIFICATION GATE: PASS")
else:
    print("MANUAL VERIFICATION GATE: NEEDS INPUT / REVISION")
    for issue in verification_issues:
        print(" -", issue)

print(
    "\nMATCH hanya berarti Agent dan manual check konsisten pada "
    "source/scope/definition yang sama. MATCH belum otomatis berarti error."
)

,finding_id,claim_from_report,dataset_source,agent_result,manual_check_id,manual_result,verification_status,evidence,finding_classification,notes
0,VF-01,order_items memiliki 3 repeated candidate keys,order_items,repeated_distinct_keys=3,MP-05,repeated_distinct_keys=3; records_under_repeat...,MATCH,"MP-05: {'ITM00130': 2, 'ITM00215': 2, 'ITM0046...",DATA_QUALITY_ISSUE,
1,VF-02,customer_segment memiliki 4 missing/blank values,customers,missing_or_blank=4; pct=1.86%,MP-07,missing_or_blank=4; pct=1.86%,MATCH,"MP-07: sample_row_index=[70, 107, 117, 173] | ...",DATA_QUALITY_ISSUE,
2,VF-03,order_items memiliki 3 exact duplicate extras,order_items,exact_duplicate_extras=3,MP-09,exact_duplicate_extras=3,MATCH,"MP-09: [{'order_item_id': 'ITM00130', 'order_i...",DATA_QUALITY_ISSUE,
3,VF-04,payment_amount memiliki 3 invalid nonblank values,payments,invalid_nonblank=3,MP-13,invalid_nonblank=3; min=-344000.0; max=4176000...,MATCH,MP-13: max_value_rows=[{'payment_id': 'PAY0247...,DATA_QUALITY_ISSUE,
4,VF-05,payment_date memiliki 0 unresolved dates,payments,unresolved_dates=0,MP-12,records=337; unresolved_dates=0; min_parsed=20...,MATCH,MP-12: representation_counts={'YYYY-MM-DD': 33...,NO_ISSUE,


MANUAL VERIFICATION GATE: PASS

MATCH hanya berarti Agent dan manual check konsisten pada source/scope/definition yang sama. MATCH belum otomatis berarti error.


## 8. Raw Integrity + Final Quality Gate

S2 selesai pada **evidence + classification**.

Tidak ada cleaning, tidak ada final join, dan raw files harus tetap identik.

In [50]:
raw_integrity_rows = []
for name, path in FILES.items():
    after_hash = file_sha256(path)
    before_hash = raw_hash_before[name]
    raw_integrity_rows.append({
        "dataset": name,
        "unchanged": before_hash == after_hash,
        "sha256_before": before_hash,
        "sha256_after": after_hash,
    })

raw_integrity_result = pd.DataFrame(raw_integrity_rows)
raw_integrity_pass = bool(raw_integrity_result["unchanged"].all())

display(raw_integrity_result)
print("RAW INTEGRITY:", "PASS" if raw_integrity_pass else "FAIL")

final_checks = {
    "Raw load + minimum schema": bool(intake_schema_pass),
    "Source structure + raw snapshot": bool(len(source_structure_result) == 4),
    "Grain & candidate key complete": bool(grain_key_gate_pass),
    "DQ Criteria confirmed before profiling": bool(DQ_CRITERIA_CONFIRMED),
    ">=10 manual profiling checks": bool(manual_profiling_gate_pass),
    "Antigravity profiling_report.md exists": bool(profiling_report_exists),
    ">=5 manual verification rows complete": bool(manual_verification_gate_pass),
    "Raw source unchanged": bool(raw_integrity_pass),
}

final_quality_gate_pass = all(final_checks.values())

display(pd.DataFrame([
    {"check": check, "status": "PASS" if passed else "NOT PASS"}
    for check, passed in final_checks.items()
]))

print()
if final_quality_gate_pass:
    print("FINAL QUALITY GATE: PASS")
    print("S2 PARTICIPANT STARTER: COMPLETE")
else:
    print("FINAL QUALITY GATE: NOT READY")
    if not grain_key_gate_pass:
        print("Mulai dari Section 3: lengkapi grain statement + candidate key.")
    elif not DQ_CRITERIA_CONFIRMED:
        print("Isi 03_DQ_Criteria lalu set DQ_CRITERIA_CONFIRMED = True.")
    elif not manual_profiling_gate_pass:
        print("Pilih dan jalankan minimal 10 manual profiling checks pada Section 5.")
    elif not profiling_report_exists:
        print("Jalankan Antigravity Brief sampai outputs/sesi_02/profiling_report.md tersedia.")
    elif not manual_verification_gate_pass:
        print("Lengkapi minimal 5 rows VERIFICATION_PLAN dari report lalu rerun Section 7.")
    elif not raw_integrity_pass:
        print("RAW INTEGRITY FAIL — hentikan proses dan restore raw package.")

,dataset,unchanged,sha256_before,sha256_after
0,orders,True,bfb422563f893805b8bfca98103e437fc395495a124e6d...,bfb422563f893805b8bfca98103e437fc395495a124e6d...
1,order_items,True,4326f3690767149d781933713c19407a19b7c0a85e4d3c...,4326f3690767149d781933713c19407a19b7c0a85e4d3c...
2,customers,True,e5d6eae9d215bbf835ddee9c87d157ec8efb6a3d35517c...,e5d6eae9d215bbf835ddee9c87d157ec8efb6a3d35517c...
3,payments,True,47d130703fd3b6c211014bb27bd255fcc645b4a771728b...,47d130703fd3b6c211014bb27bd255fcc645b4a771728b...


RAW INTEGRITY: PASS


,check,status
0,Raw load + minimum schema,PASS
1,Source structure + raw snapshot,PASS
2,Grain & candidate key complete,PASS
3,DQ Criteria confirmed before profiling,PASS
4,>=10 manual profiling checks,PASS
5,Antigravity profiling_report.md exists,PASS
6,>=5 manual verification rows complete,PASS
7,Raw source unchanged,PASS



FINAL QUALITY GATE: PASS
S2 PARTICIPANT STARTER: COMPLETE


### Definition of Done — Practice in Class S2

Sebelum selesai:
- 4 physical files berhasil dibaca menjadi 4 logical datasets;
- `01_Source_Structure` terisi;
- grain + candidate key untuk 4 datasets diuji;
- DQ Criteria ditetapkan **sebelum** profiling utama;
- minimal 10 manual profiling checks dijalankan;
- Antigravity menghasilkan `outputs/sesi_02/profiling_report.md`;
- minimal 5 claims dari report diverifikasi manual;
- classification S2 memakai `Data Quality Issue / Valid but Unusual / Ambiguous / No Issue`;
- raw files tetap unchanged;
- `FINAL QUALITY GATE: PASS`.

Mentor boleh memberi **reasoning/hint**, tetapi tidak perlu mengirim profiling query/code tambahan.